In [ ]:
import random
import collections
from datasets import load_dataset

# ======================================================================================
# 💖✨ 데이터셋 분석 주제: 나만의 AI 비서 만들기! (Instruction/Intent Analysis)
# 📝 데이터셋 설명: 'youjunhyeok/Magpie-Llama-3.1-Pro-500K-Filtered-ko'
# 💡 이 데이터셋은 한국어로 된 다양한 질문-답변(Instruction/Response) 쌍으로 이루어져 있습니다.
#    어떤 질문(Instruction)이 들어왔을 때, 어떤 의도(Intent)로, 어떤 난이도(Difficulty)의 답변이 필요했는지 분석하는 과정입니다.
#    AI가 더 똑똑한 비서가 되기 위해 '질문 패턴'과 '답변 전략'을 배우는 것처럼요!
# 🎯 목표: 데이터를 읽고, 가장 많이 등장하는 사용자 의도를 찾아내고, 새로운 프롬프트가 어떻게 만들어지는지 시뮬레이션해봅니다!
# ======================================================================================

DATASET_NAME = "youjunhyeok/Magpie-Llama-3.1-Pro-500K-Filtered-ko"
SAMPLE_COUNT = 100

# ------------------------- 1. 데이터 로드 (지능형 로딩 시스템) -------------------------
print("🌟 1. 데이터셋 로딩을 시작합니다...")
dataset = None

try:
    # ➡️ 1단계: 스트리밍 로드 시도 (메모리 효율성 최우선!)
    print("➡️ 스트리밍(Streaming) 모드로 데이터셋을 로드하는 것을 시도합니다. (빠른 테스트에 최적)")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 데이터셋을 스트리밍 모드로 로드했습니다. 대용량 데이터 처리에 강합니다!")
except Exception as e:
    # ➡️ 2단계: 스트리밍 실패 시 일반 로드 (안정성 확보)
    print(f"⚠️ 경고: 스트리밍 로드 실패 또는 연결 문제 발생 ({e}). 일반 모드로 전환합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='train')
        print("✅ 성공! 데이터셋을 일반(Non-streaming) 모드로 로드했습니다. 안정성이 높습니다.")
    except Exception as e_fallback:
        print(f"🚨 치명적 오류: 데이터셋 로드에 실패했습니다. 네트워크 또는 데이터셋 ID를 확인해주세요. ({e_fallback})")
        exit()


# ------------------------- 2. 샘플 데이터 준비 (메모리 절약 필수!) -------------------------
print("\n✨ 2. 분석에 필요한 상위 샘플 데이터", SAMPLE_COUNT, "개를 추출합니다...")

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋 (IterableDataset)
    # 이 경우, 메모리에 전체를 올리지 않고 상위 K개만 반복합니다.
    print("🔗 스트리밍 모드 감지: .take() 패턴을 사용합니다.")
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
    # iter()를 통해 반드시 이터레이터로 변환합니다.
    sampled_dataset_list = list(sampled_dataset_iterator) 
else:
    # 일반 데이터셋 (Dataset)
    print("📚 일반 데이터셋 감지: .select() 또는 list() 패턴을 사용합니다.")
    # 안전하게 상위 K개만 리스트로 만듭니다.
    sampled_dataset_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))


# ------------------------- 3. 핵심 데이터 분석: '의도'와 '난이도' 탐색 -------------------------

print("\n🔍 3. [분석 챌린지] 사용자 의도(Intent)와 난이도(Difficulty)를 카운트합니다!")

# 💥 핵심 필드 추출: intent와 difficulty만 뽑아냅니다.
intents = []
difficulties = []

for sample in sampled_dataset_list:
    # 안전하게 키가 존재하는지 확인합니다.
    if 'intent' in sample and sample['intent']:
        intents.append(sample['intent'])
    if 'difficulty' in sample and sample['difficulty']:
        difficulties.append(sample['difficulty'])

# 📊 빈도수 계산 (가장 많이 쓰이는 주제를 찾아요!)
intent_counts = collections.Counter(intents)
difficulty_counts = collections.Counter(difficulties)


print("\n=====================================================================================")
print("🧠 분석 결과: AI 비서의 주요 학습 포인트")
print("=====================================================================================")

print(f"\n✅ 분석한 데이터 샘플 수: {len(sampled_dataset_list)} 건")

# 💡 Intent 분석
print("\n[🔍 1. 사용자 의도 (Intent) 분석]")
print("-> 가장 많이 요청받은 의도가 무엇일까요? AI가 먼저 학습해야 할 주제입니다!")
top_intents = intent_counts.most_common(3)
if top_intents:
    for intent, count in top_intents:
        print(f"    - '{intent}' 의도: {count}회 (가장 많이 등장하는 패턴!)")
else:
    print("    - Intent 데이터를 찾을 수 없거나 비어있습니다.")


# 💡 Difficulty 분석
print("\n[💪 2. 난이도 (Difficulty) 분포 분석]")
print("-> 데이터는 다양한 난이도를 포괄합니다. 난이도별로 답변 스타일을 다르게 훈련해야 합니다.")
top_difficulties = difficulty_counts.most_common(3)
if top_difficulties:
    for difficulty, count in top_difficulties:
        print(f"    - '{difficulty}' 난이도: {count}회")
else:
    print("    - Difficulty 데이터를 찾을 수 없거나 비어있습니다.")

# ------------------------- 4. 실습: Prompt 생성 시뮬레이션 -------------------------

print("\n\n✨ 4. [미니 실습] 데이터를 프롬프트로 재구성해보기!")
print("------------------------------------------------------------------------------------")

def generate_prompt(sample):
    """
    주어진 데이터 샘플을 LLM이 이해하기 쉬운 구조화된 프롬프트로 변환합니다.
    이 과정이 바로 '프롬프트 엔지니어링'의 핵심입니다!
    """
    
    instruction = sample.get('instruction', '---')
    response = sample.get('response', '---')
    intent = sample.get('intent', 'N/A')
    difficulty = sample.get('difficulty', 'N/A')
    
    prompt = f"""
    [📝 사용자 지시 (Instruction)]
    {instruction}
    
    [💡 분석된 의도 (Intent)]
    의도: {intent}
    난이도: {difficulty}
    
    [🧠 기대하는 응답 (Expected Response)]
    {response}
    """
    return prompt.strip()

# 임의로 2개의 샘플을 골라 프롬프트 생성 과정을 보여줍니다.
print("📌 샘플 1: 일반적인 질문 케이스 분석 (가장 첫 번째 샘플)")
if sampled_dataset_list:
    sample_1 = sampled_dataset_list[0]
    prompt_1 = generate_prompt(sample_1)
    print("=====================================================================================")
    print("<<✨ AI 모델에게 전달할 최종 형식 (프롬프트 예시) >>")
    print(prompt_1)
    print("=====================================================================================")


print("\n\n📌 샘플 2: 다른 패턴의 질문 케이스 분석 (첫 번째 샘플과 다른 하나)")
if len(sampled_dataset_list) > 1:
    sample_2 = sampled_dataset_list[1]
    prompt_2 = generate_prompt(sample_2)
    print("=====================================================================================")
    print("<<✨ AI 모델에게 전달할 최종 형식 (프롬프트 예시) >>")
    print(prompt_2)
    print("=====================================================================================")

print("\n🌟🎉 축하합니다! 데이터셋을 로드하고, 분석하고, 실제 사용 목적에 맞춰 가공하는 전 과정을 완벽하게 마쳤습니다.")
print("다음 단계는 이 분석을 바탕으로 '최적의 프롬프트 템플릿'을 만드는 것입니다. 파이팅!");